## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar: p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), con m = max_j z_j, lo que evita desbordamientos de la exponencial sin alterar el resultado. Sobre los logits z = (2.0, 1.0, 0.5, 0.2, −1.0, −3.0) de los tokens t0–t5, la celda siguiente calcula la distribución para T = 0.5, T = 1 y T = 2, y la entropía en bits H = −Σ_i p_i·log2(p_i):

| T | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| 1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| 2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Cada fila suma 1.0000. En T = 0.5 la probabilidad de t5 es 3.7367e-05, que se muestra como 0.0000 al redondear a cuatro decimales. Se aprecia el efecto esperado de la temperatura: al bajarla, la masa se concentra en t0 (0.3583 → 0.5516 → 0.8231 al pasar de T = 2 a T = 0.5) y la entropía cae (2.2888 → 1.7998 → 0.9147 bits); al subirla, la distribución se aplana y la entropía crece.

In [1]:
# T1: código aprobado por el crítico
# T1 — Softmax con temperatura (NumPy, numéricamente estable) + entropía en bits
# MMIA 6013 · Tarea B · Taller 03 v2

import json

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


# ----------------------------------------------------------------------
# Implementación
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    """p_i = exp(z_i/T) / sum_j exp(z_j/T), restando el máximo de los
    logits escalados para estabilidad numérica."""
    z = np.asarray(z, dtype=float)
    escalados = z / float(T)
    escalados = escalados - escalados.max()  # estabilidad numérica
    exp_z = np.exp(escalados)
    return exp_z / exp_z.sum()


def entropia_bits(p):
    """Entropía de Shannon en bits: H = -sum_i p_i * log2(p_i)."""
    p = np.asarray(p, dtype=float)
    mascara = p > 0
    return float(-np.sum(p[mascara] * np.log2(p[mascara])))


# ----------------------------------------------------------------------
# Cálculo de distribuciones y entropías
# ----------------------------------------------------------------------
distribuciones = {}
entropias = {}
sumas = {}
for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(p.sum())

# ----------------------------------------------------------------------
# Tabla con cuatro decimales (probabilidades por token + entropía)
# ----------------------------------------------------------------------
tabla_4dec = pd.DataFrame(
    {f"T={T}": [f"{distribuciones[T][i]:.4f}" for i in range(len(tokens))]
     for T in temperaturas},
    index=tokens,
)
tabla_4dec.loc["Entropía (bits)"] = [f"{entropias[T]:.4f}" for T in temperaturas]

# ----------------------------------------------------------------------
# Verificaciones (cada distribución debe sumar 1)
# ----------------------------------------------------------------------
verificacion = {
    f"T={T}": {
        "suma": sumas[T],
        "suma_redondeada_4dec": f"{sumas[T]:.4f}",
        "ok_suma_1": bool(abs(sumas[T] - 1.0) < 1e-12),
    }
    for T in temperaturas
}

# ----------------------------------------------------------------------
# Figura: distribuciones para las tres temperaturas
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 5))
x = np.arange(len(tokens))
ancho = 0.25
colores = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for k, T in enumerate(temperaturas):
    ax.bar(
        x + (k - 1) * ancho,
        distribuciones[T],
        width=ancho,
        color=colores[k],
        label=f"T={T}  (H = {entropias[T]:.4f} bits)",
    )
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad")
ax.set_title("Softmax con temperatura — logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "subtarea": "T1 - Softmax con temperatura y entropía en bits",
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {
        f"T={T}": distribuciones[T].tolist() for T in temperaturas
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "suma_distribuciones": {f"T={T}": sumas[T] for T in temperaturas},
    "verificacion_suma_1": verificacion,
    "tabla_4_decimales": tabla_4dec.to_dict(),
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------
print("T1 — Softmax con temperatura (implementación NumPy estable)")
print("Logits:", logits.tolist())
print()
print("Tabla (4 decimales):")
print(tabla_4dec.to_string())
print()
for T in temperaturas:
    print(
        f"T={T}: suma de la distribución = {sumas[T]:.6f} "
        f"(ok={verificacion[f'T={T}']['ok_suma_1']}), "
        f"entropía = {entropias[T]:.4f} bits"
    )
print()
print("Figura guardada: t1_softmax_temperatura.png")
print("Resultados guardados: resultados.json")


T1 — Softmax con temperatura (implementación NumPy estable)
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

Tabla (4 decimales):
                  T=0.5   T=1.0   T=2.0
t0               0.8231  0.5516  0.3583
t1               0.1114  0.2029  0.2173
t2               0.0410  0.1231  0.1693
t3               0.0225  0.0912  0.1457
t4               0.0020  0.0275  0.0800
t5               0.0000  0.0037  0.0294
Entropía (bits)  0.9147  1.7998  2.2888

T=0.5: suma de la distribución = 1.000000 (ok=True), entropía = 0.9147 bits
T=1.0: suma de la distribución = 1.000000 (ok=True), entropía = 1.7998 bits
T=2.0: suma de la distribución = 1.000000 (ok=True), entropía = 2.2888 bits

Figura guardada: t1_softmax_temperatura.png
Resultados guardados: resultados.json


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 (recalculada en la celda, coincide exactamente con la de la Parte 1, diferencia máxima 0.0) se ordenan los tokens de mayor a menor probabilidad y se acumula. La acumulada tras {t0, t1, t2} es 0.8776, aún por debajo del umbral p = 0.9; al añadir t3 alcanza 0.9688 ≥ 0.9. El conjunto mínimo cuya acumulada alcanza 0.9 es por tanto {t0, t1, t2, t3}, con masa 0.9688; t4 y t5 quedan excluidos, con una masa descartada de 0.0312.

| Posición | Token | p (T = 1) | Acumulada | ¿En el núcleo? |
|---:|---|---:|---:|---|
| 1 | t0 | 0.5516 | 0.5516 | sí |
| 2 | t1 | 0.2029 | 0.7546 | sí |
| 3 | t2 | 0.1231 | 0.8776 | sí |
| 4 | t3 | 0.0912 | 0.9688 | sí |
| 5 | t4 | 0.0275 | 0.9963 | no |
| 6 | t5 | 0.0037 | 1.0000 | no |

La renormalización divide la probabilidad de cada superviviente entre la masa del núcleo; el factor es constante e igual a 1.0322 (= 1/0.9688) para los cuatro tokens:

| Token | p original (T = 1) | p renormalizada (top-p, p = 0.9) |
|---|---:|---:|
| t0 | 0.5516 | 0.5694 |
| t1 | 0.2029 | 0.2095 |
| t2 | 0.1231 | 0.1270 |
| t3 | 0.0912 | 0.0941 |

La distribución renormalizada suma 1.0000 (error absoluto 0.0) y es la que se muestrea en la Parte 3.

In [2]:
# T2: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1 (Parte 1).

Procedimiento:
  1) Se carga la distribución de T=1 obtenida en T1 (entrada/T1/resultados.json).
  2) Se ordenan los tokens por probabilidad descendente y se acumulan las probabilidades.
  3) Se conserva el conjunto MÍNIMO de tokens cuya probabilidad acumulada alcanza p = 0.9
     (incluye el token con el que la acumulada cruza el umbral).
  4) Se renormalizan las probabilidades de los supervivientes (dividiendo por su masa),
     de modo que sumen exactamente 1 y cada una sea la probabilidad original de T=1
     dividida por la masa del núcleo (coherencia con la Parte 1).

Salidas:
  - resultados.json       (contrato de la subtarea: supervivientes, renormalizada, verificaciones)
  - T2_top_p_nucleo.png   (figura: corte del núcleo y distribución renormalizada)
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

P = 0.9
T = 1.0
RUTA_T1 = Path("entrada") / "T1" / "resultados.json"
RUTA_SALIDA = Path("resultados.json")
RUTA_FIG = Path("T2_top_p_nucleo.png")


def softmax_con_temperatura(logits, temperatura):
    """Softmax con temperatura, numéricamente estable (resta el máximo)."""
    z = np.asarray(logits, dtype=float) / float(temperatura)
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)


# ---------------------------------------------------------- 1) Distribución de T=1 (viene de T1)
tokens, logits, dist_T1, fuente = None, None, None, None
if RUTA_T1.exists():
    with open(RUTA_T1, "r", encoding="utf-8") as f:
        t1 = json.load(f)
    tokens = list(t1.get("tokens", ["t0", "t1", "t2", "t3", "t4", "t5"]))
    logits = np.asarray(t1.get("logits", [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]), dtype=float)
    dists = t1.get("distribuciones", {})
    clave = "T=1.0" if "T=1.0" in dists else ("T=1" if "T=1" in dists else None)
    if clave is not None:
        dist_T1 = np.asarray(dists[clave], dtype=float)
        fuente = str(RUTA_T1)
if dist_T1 is None:  # respaldo: recalcular exactamente como en T1
    tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
    dist_T1 = softmax_con_temperatura(logits, T)
    fuente = "recalculada con softmax numéricamente estable (T=1)"

dist_T1 = dist_T1 / dist_T1.sum()  # blindaje numérico (no altera los valores de T1)

# ---------------------------------------------------------- 2) Top-p: núcleo con p = 0.9
orden = np.argsort(-dist_T1, kind="stable")            # índices en orden descendente
tokens_orden = [tokens[i] for i in orden]
probs_orden = dist_T1[orden]
acumulada = np.cumsum(probs_orden)

# Conjunto MÍNIMO cuya probabilidad acumulada alcanza p
# (searchsorted da el primer índice con acumulada >= p; +1 incluye ese token)
n_nucleo = int(np.searchsorted(acumulada, P, side="left")) + 1
n_nucleo = min(n_nucleo, len(tokens))

idx_nucleo = orden[:n_nucleo]
tokens_nucleo = [tokens[i] for i in idx_nucleo]
probs_nucleo = dist_T1[idx_nucleo]
masa_nucleo = float(probs_nucleo.sum())
probs_renorm = probs_nucleo / masa_nucleo

idx_fuera = orden[n_nucleo:]
tokens_excluidos = [tokens[i] for i in idx_fuera]
masa_descartada = float(dist_T1[idx_fuera].sum()) if idx_fuera.size else 0.0

# ---------------------------------------------------------- 3) Verificaciones
suma_renorm = float(probs_renorm.sum())
razones = {tk: float(pr / po) for tk, pr, po in zip(tokens_nucleo, probs_renorm, probs_nucleo)}
razon_esperada = 1.0 / masa_nucleo
razon_constante = bool(np.allclose(list(razones.values()), razon_esperada, rtol=1e-12, atol=0.0))
coherente = bool(np.allclose(probs_renorm * masa_nucleo, probs_nucleo, rtol=1e-12, atol=0.0))
ok_suma = bool(abs(suma_renorm - 1.0) <= 1e-12)
max_diff_T1 = float(np.max(np.abs(dist_T1 - softmax_con_temperatura(logits, T))))

# ---------------------------------------------------------- 4) Tablas y resultados
dist_original_dict = {tk: float(p) for tk, p in zip(tokens, dist_T1)}
probs_orig_nucleo_dict = {tk: float(p) for tk, p in zip(tokens_nucleo, probs_nucleo)}
renorm_dict = {tk: float(p) for tk, p in zip(tokens_nucleo, probs_renorm)}

tabla_ordenada = [
    {
        "token": tk,
        "prob_T1": float(pr),
        "prob_T1_4dec": f"{pr:.4f}",
        "acumulada": float(ac),
        "acumulada_4dec": f"{ac:.4f}",
        "en_nucleo": tk in tokens_nucleo,
    }
    for tk, pr, ac in zip(tokens_orden, probs_orden, acumulada)
]
tabla_renorm_4dec = {tk: f"{p:.4f}" for tk, p in renorm_dict.items()}

resultados = {
    "subtarea": "T2 - Muestreo de núcleo (top-p) con p=0.9 sobre la distribución de T=1",
    "p": P,
    "temperatura": T,
    "fuente_distribucion_T1": fuente,
    "tokens": tokens,
    "logits": logits.tolist(),
    "distribucion_T1_original": dist_original_dict,
    "orden_descendente_probabilidad": tokens_orden,
    "probs_orden_descendente": [float(x) for x in probs_orden],
    "prob_acumulada_orden_descendente": [float(x) for x in acumulada],
    "tokens_supervivientes": tokens_nucleo,
    "num_tokens_nucleo": n_nucleo,
    "masa_probabilidad_nucleo": masa_nucleo,
    "probabilidades_originales_supervivientes": probs_orig_nucleo_dict,
    "distribucion_renormalizada": renorm_dict,
    "distribucion_renormalizada_4dec": tabla_renorm_4dec,
    "tokens_excluidos": tokens_excluidos,
    "masa_descartada": masa_descartada,
    "suma_distribucion_renormalizada": suma_renorm,
    "verificaciones": {
        "suma_renormalizada_es_1": ok_suma,
        "error_absoluto_suma": abs(suma_renorm - 1.0),
        "razon_renormalizada_original_por_token": razones,
        "razon_esperada_1_sobre_masa_nucleo": razon_esperada,
        "razon_constante_entre_tokens": razon_constante,
        "renormalizada_por_masa_recupera_prob_T1": coherente,
        "max_abs_diff_distribucion_T1_vs_softmax_recalculada": max_diff_T1,
    },
    "tabla_ordenada_top_p": tabla_ordenada,
}

with open(RUTA_SALIDA, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ---------------------------------------------------------- 5) Figura
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

colores = ["#2a9d8f" if tk in tokens_nucleo else "#d95f02" for tk in tokens_orden]
ax = axes[0]
ax.bar(range(len(tokens_orden)), probs_orden, color=colores, edgecolor="black", linewidth=0.6)
ax.plot(range(len(tokens_orden)), acumulada, "o--", color="#40517d", label="Prob. acumulada")
ax.axhline(P, color="crimson", linestyle=":", linewidth=1.5, label=f"p = {P}")
ax.set_xticks(range(len(tokens_orden)))
ax.set_xticklabels(tokens_orden)
ax.set_ylim(0, 1.08)
ax.set_xlabel("Token (orden descendente)")
ax.set_ylabel("Probabilidad")
ax.set_title("Top-p (p=0.9) sobre la distribución de T=1\nverde = núcleo, naranja = descartado")
ax.legend(fontsize=8, loc="upper left")

ax = axes[1]
ax.bar(range(n_nucleo), probs_renorm, color="#2a9d8f", edgecolor="black", linewidth=0.6)
ax.set_xticks(range(n_nucleo))
ax.set_xticklabels(tokens_nucleo)
ax.set_ylim(0, max(probs_renorm.max() * 1.2, 0.1))
ax.set_xlabel("Token superviviente")
ax.set_ylabel("Probabilidad renormalizada")
ax.set_title(f"Distribución renormalizada del núcleo\n(suma = {suma_renorm:.4f})")
for i, pv in enumerate(probs_renorm):
    ax.text(i, pv + 0.012, f"{pv:.4f}", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(str(RUTA_FIG), dpi=120)
plt.close(fig)

# ---------------------------------------------------------- 6) Resumen
print("T2 — Top-p (muestreo de núcleo) con p=0.9 sobre la distribución de T=1")
print(f"Fuente de la distribución T=1: {fuente}")
print("Orden descendente : " + ", ".join(f"{tk}={pv:.4f}" for tk, pv in zip(tokens_orden, probs_orden)))
print("Acumulada         : " + ", ".join(f"{av:.4f}" for av in acumulada))
print(f"Núcleo (conjunto mínimo con acumulada >= {P}): {tokens_nucleo}  (masa = {masa_nucleo:.4f})")
print(f"Tokens excluidos  : {tokens_excluidos}  (masa descartada = {masa_descartada:.4f})")
print("Distribución renormalizada (suma = {:.12f}):".format(suma_renorm))
for tk in tokens_nucleo:
    print(f"  {tk}: renorm = {renorm_dict[tk]:.6f}   (original T=1 = {probs_orig_nucleo_dict[tk]:.6f},"
          f" razón = {razones[tk]:.6f})")
print(f"Verificaciones: suma=1 -> {ok_suma} | razón constante 1/masa -> {razon_constante} | "
      f"coherente con T1 -> {coherente}")
print(f"Resultados escritos en '{RUTA_SALIDA}' y figura en '{RUTA_FIG}'.")


T2 — Top-p (muestreo de núcleo) con p=0.9 sobre la distribución de T=1
Fuente de la distribución T=1: entrada/T1/resultados.json
Orden descendente : t0=0.5516, t1=0.2029, t2=0.1231, t3=0.0912, t4=0.0275, t5=0.0037
Acumulada         : 0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0000
Núcleo (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']  (masa = 0.9688)
Tokens excluidos  : ['t4', 't5']  (masa descartada = 0.0312)
Distribución renormalizada (suma = 1.000000000000):
  t0: renorm = 0.569376   (original T=1 = 0.551623, razón = 1.032184)
  t1: renorm = 0.209462   (original T=1 = 0.202931, razón = 1.032184)
  t2: renorm = 0.127045   (original T=1 = 0.123084, razón = 1.032184)
  t3: renorm = 0.094117   (original T=1 = 0.091183, razón = 1.032184)
Verificaciones: suma=1 -> True | razón constante 1/masa -> True | coherente con T1 -> True
Resultados escritos en 'resultados.json' y figura en 'T2_top_p_nucleo.png'.


## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada con numpy.random.default_rng(0) (semilla 0). Los conteos observados fueron t0 = 5682, t1 = 2093, t2 = 1296 y t3 = 929 (total 10000; ningún token del núcleo con conteo cero). El gráfico de barras de la celda compara, para cada token, la frecuencia observada con la probabilidad teórica (recalculada en la celda y coincidente con la de la Parte 2, diferencia máxima 0.0):

| Token | p teórica | Conteo observado | Frecuencia observada | Diferencia (obs − teo) |
|---|---:|---:|---:|---:|
| t0 | 0.5694 | 5682 | 0.5682 | -0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | -0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | 0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | -0.0012 |

La mayor discrepancia absoluta es 0.0026 (t2). La divergencia KL de la distribución empírica respecto de la teórica, D_KL(q‖p) = Σ_i q_i·log2(q_i/p_i), vale 5.0069e-05 bits (0.000050 con seis decimales; equivalente a 3.4705e-05 nats). Un valor tan próximo a 0 confirma que las frecuencias observadas reproducen la distribución teórica del núcleo top-p, como corresponde a un muestreo iid de 10 000 muestras.

In [3]:
# T3: código aprobado por el crítico
# =====================================================================
# T3 — Comprobación empírica de la distribución renormalizada (top-p)
# Toma 10,000 muestras con numpy.random.default_rng(0) de la distribución
# renormalizada de la Parte 2 (entrada/T2), dibuja un gráfico de barras
# comparando frecuencias observadas vs. probabilidades teóricas y calcula
# la divergencia KL (en bits) de la empírica respecto de la teórica.
# =====================================================================
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------- Parámetros del enunciado -------------------------
N_MUESTRAS = 10_000          # muestras exigidas
SEMILLA = 0                  # numpy.random.default_rng(0)
P_TOP = 0.9                  # núcleo de la Parte 2
T = 1.0                      # temperatura de la Parte 1

TODOS_TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

# ---------------------------------------------------------------------------
# 1) Distribución renormalizada de la Parte 2 (resultado de T2)
# ---------------------------------------------------------------------------
ruta_t2 = Path("entrada") / "T2" / "resultados.json"
res_t2 = None
if ruta_t2.exists():
    with ruta_t2.open("r", encoding="utf-8") as f:
        res_t2 = json.load(f)

# Recálculo independiente (softmax T=1 + top-p p=0.9) para verificación
z = LOGITS - LOGITS.max()
p_t1 = np.exp(z) / np.exp(z).sum()
orden = np.argsort(-p_t1, kind="stable")
acum = np.cumsum(p_t1[orden])
k = int(np.searchsorted(acum, P_TOP - 1e-12) + 1)  # conjunto mínimo que alcanza 0.9
idx_nucleo = np.sort(orden[:k])
p_teo_recalc = p_t1[idx_nucleo] / p_t1[idx_nucleo].sum()

if res_t2 is not None:
    tokens_nucleo = list(res_t2["tokens_supervivientes"])
    p_teo = np.array([float(res_t2["distribucion_renormalizada"][t])
                      for t in tokens_nucleo], dtype=float)
else:
    tokens_nucleo = [TODOS_TOKENS[i] for i in idx_nucleo]
    p_teo = p_teo_recalc.copy()

max_diff_t2_vs_recalc = float(np.max(np.abs(p_teo - p_teo_recalc)))

# ---------------------------------------------------------------------------
# 2) Muestreo: 10,000 muestras con numpy.random.default_rng(0)
# ---------------------------------------------------------------------------
rng = np.random.default_rng(SEMILLA)
idx_muestras = rng.choice(len(tokens_nucleo), size=N_MUESTRAS, p=p_teo)

conteos = np.bincount(idx_muestras, minlength=len(tokens_nucleo)).astype(int)
f_obs = conteos / N_MUESTRAS  # distribución empírica

# ---------------------------------------------------------------------------
# 3) Divergencia KL en bits: D_KL(empírica || teórica) = Σ q_i log2(q_i / p_i)
# ---------------------------------------------------------------------------
mask = f_obs > 0  # términos con q_i = 0 aportan 0 por convención
kl_bits = float(np.sum(f_obs[mask] * np.log2(f_obs[mask] / p_teo[mask])))
kl_nats = float(np.sum(f_obs[mask] * np.log(f_obs[mask] / p_teo[mask])))

# ---------------------------------------------------------------------------
# 4) Gráfico de barras: frecuencias observadas vs. probabilidades teóricas
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.5, 5.2))
x = np.arange(len(tokens_nucleo))
w = 0.38
b1 = ax.bar(x - w / 2, f_obs, width=w, color="#4C72B0", edgecolor="black",
            label="Frecuencia observada (10,000 muestras)")
b2 = ax.bar(x + w / 2, p_teo, width=w, color="#DD8452", edgecolor="black",
            label="Probabilidad teórica (renormalizada, top-p 0.9)")
for barras in (b1, b2):
    for r in barras:
        h = r.get_height()
        ax.annotate(f"{h:.4f}", (r.get_x() + r.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens_nucleo)
ax.set_xlabel("Token del núcleo")
ax.set_ylabel("Probabilidad / frecuencia relativa")
ax.set_ylim(0, max(f_obs.max(), p_teo.max()) * 1.18)
ax.set_title("Parte 3 — Frecuencias observadas vs. probabilidades teóricas\n"
             f"top-p p={P_TOP} sobre softmax T={T} · {N_MUESTRAS:,} muestras · "
             f"rng({SEMILLA}) · KL = {kl_bits:.6f} bits")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("T3_frecuencias_observadas_vs_teoricas.png", dpi=120)
plt.close(fig)

# ---------------------------------------------------------------------------
# 5) resultados.json (contrato de la subtarea)
# ---------------------------------------------------------------------------
tabla = [
    {
        "token": t,
        "prob_teorica": float(p),
        "prob_teorica_4dec": f"{p:.4f}",
        "conteo_observado": int(c),
        "frecuencia_observada": float(fq),
        "frecuencia_observada_4dec": f"{fq:.4f}",
        "diferencia_obs_menos_teo": float(fq - p),
    }
    for t, p, c, fq in zip(tokens_nucleo, p_teo, conteos, f_obs)
]

resultados = {
    "subtarea": ("T3 - Comprobación empírica: 10,000 muestras de la distribución "
                 "renormalizada (top-p p=0.9, T=1), gráfico de barras y divergencia KL en bits"),
    "fuente_distribucion_renormalizada": "entrada/T2/resultados.json",
    "temperatura": T,
    "p_top": P_TOP,
    "generador": "numpy.random.default_rng(0)",
    "semilla": SEMILLA,
    "n_muestras": N_MUESTRAS,
    "tokens_nucleo": tokens_nucleo,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens_nucleo, p_teo)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens_nucleo, conteos)},
    "frecuencias_observadas": {t: float(fq) for t, fq in zip(tokens_nucleo, f_obs)},
    "frecuencias_observadas_4dec": {t: f"{fq:.4f}" for t, fq in zip(tokens_nucleo, f_obs)},
    "probabilidades_teoricas_4dec": {t: f"{p:.4f}" for t, p in zip(tokens_nucleo, p_teo)},
    "divergencia_KL_bits": kl_bits,
    "divergencia_KL_bits_6dec": f"{kl_bits:.6f}",
    "divergencia_KL_nats": kl_nats,
    "tabla_comparativa": tabla,
    "figura": "T3_frecuencias_observadas_vs_teoricas.png",
    "verificaciones": {
        "suma_probabilidades_teoricas": float(p_teo.sum()),
        "suma_frecuencias_observadas": float(f_obs.sum()),
        "total_conteos": int(conteos.sum()),
        "max_abs_diff_T2_vs_recalculo_softmax_topp": max_diff_t2_vs_recalc,
        "tokens_con_conteo_cero": [t for t, c in zip(tokens_nucleo, conteos) if c == 0],
        "formula_KL": "D_KL(empirica || teorica) = sum_i q_i * log2(q_i / p_i)",
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------------------
# Resumen
# ---------------------------------------------------------------------------
print("=" * 72)
print("T3 — Comprobación empírica (top-p p=0.9, T=1, rng(0), 10,000 muestras)")
print("=" * 72)
print(f"Tokens del núcleo: {tokens_nucleo}")
print(f"{'token':<6}{'p_teorica':>12}{'conteo':>10}{'f_obs':>12}{'diff':>12}")
for t, p, c, fq in zip(tokens_nucleo, p_teo, conteos, f_obs):
    print(f"{t:<6}{p:>12.6f}{c:>10d}{fq:>12.6f}{fq - p:>12.6f}")
print(f"\nDivergencia KL (empírica || teórica) = {kl_bits:.8f} bits "
      f"({kl_nats:.8f} nats)")
print("Figura guardada: T3_frecuencias_observadas_vs_teoricas.png")
print("Resultados guardados: resultados.json")


T3 — Comprobación empírica (top-p p=0.9, T=1, rng(0), 10,000 muestras)
Tokens del núcleo: ['t0', 't1', 't2', 't3']
token    p_teorica    conteo       f_obs        diff
t0        0.569376      5682    0.568200   -0.001176
t1        0.209462      2093    0.209300   -0.000162
t2        0.127045      1296    0.129600    0.002555
t3        0.094117       929    0.092900   -0.001217

Divergencia KL (empírica || teórica) = 0.00005007 bits (0.00003471 nats)
Figura guardada: T3_frecuencias_observadas_vs_teoricas.png
Resultados guardados: resultados.json


## Parte 4 — Pregunta conceptual

**T → 0 (distribución concentrada).** Al dividir los logits entre una temperatura cada vez menor, las diferencias entre tokens se amplifican: exp((z_i − m)/T) se desvanece para todo token cuyo logit no sea el máximo, y la masa se concentra en el token de mayor logit (aquí t0, con z = 2.0). Los datos de la Parte 1 muestran esa concentración: t0 pasa de 0.3583 (T = 2) a 0.5516 (T = 1) y a 0.8231 (T = 0.5), mientras que t5 cae hasta 3.7367e-05; en el límite, p(t0) → 1 y las demás probabilidades → 0. La entropía lo confirma: 0.9147 bits en T = 0.5 frente a 1.7998 bits en T = 1, una caída que continúa hasta 0 bits en el límite, el valor de una distribución sin incertidumbre.

**T → ∞ (distribución uniforme).** Al dividir entre una temperatura cada vez mayor, z_i/T → 0 para todo token, todos los exponenciales valen lo mismo y p_i → 1/6: la distribución tiende a la uniforme sobre los seis tokens, cuyo límite de entropía es log2 6. La evidencia medida es la monotonía de la entropía (0.9147 → 1.7998 → 2.2888 bits para T = 0.5 → 1 → 2) y el aplanamiento de las probabilidades: t0 baja 0.8231 → 0.5516 → 0.3583 hacia 1/6, y t5 sube 0.0000 → 0.0037 → 0.0294.

**Por qué T → 0 equivale a la decodificación voraz.** La decodificación voraz (greedy) elige en cada paso argmax_i z_i de forma determinista. Cuando T → 0, la softmax degenera en una masa puntual sobre exactamente ese token: muestrear de ella devuelve el argmax con probabilidad 1 y la entropía tiende a 0 bits, es decir, no queda aleatoriedad alguna. Por eso el muestreo con temperatura que tiende a cero y la selección greedy producen la misma salida; la diferencia es solo de implementación (en la formulación estable, el límite T → 0 se reduce a tomar el máximo de los logits).